<div align='center'>
<h1 style='font-size:38px;'>Grain Temperature Calculation</h1>
<h2 style='font-weight:400;'>Using <code>GrainStar</code> to derive equilibrium temperatures</h2>
<p><em>Thermal distance grid → temperature vs distance & size</em></p>
</div>

---
**Contents:**
1. Initialize grain and star
2. Load thermal distance grid
3. Examine temperature range
4. Interpolate temperatures
5. Plot profiles for selected sizes
6. Contour map size vs distance

## 1. Imports and object setup

In [ ]:
from pyGrater import Star, Grain, Temperature
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Create grain (loads Q tables) and star (loads spectrum + properties)
grain = Grain(redo_Q=False, composition='astroSi')
star = Star(star_name='HD113766')

# Initialize Temperature (loads or computes thermal distance grid)
temperature = Temperature(grain, star, N_temp=1000, redo_therm_dist=False)

## 2. Thermal equilibrium distance
To calculate the temperature of dust grains, we first calculate the thermal equilibrium distance, $r(a,T_{\mathrm{d}})$, for a range of grain sizes and temperatures (i.e the distance at which a grain of certain size and temperature would be at thermal equilibrium).

It is calculated in the following way:
\begin{equation}
r(a,T_{\mathrm{d}})=\frac{d_{\star}}{2}\sqrt{\frac{\int_{\lambda}Q_{\mathrm{abs}}F_{\star}(\lambda)\mathrm{d}\lambda}{\int_{\lambda}Q_{\mathrm{abs}}\pi B_{\lambda}\left(T_{\mathrm{d}}\right)\mathrm{d}\lambda}}
\end{equation}
where F⋆(λ) is the stellar flux at Earth. 

In [ ]:
therm_dist = temperature.therm_dist          # shape: (n_sizes, n_T)
temp_range = temperature.temp_range         # temperatures (K)
sizes = grain.Qabs_sizes                  # micron
print('Thermal distance grid shape:', therm_dist.shape)
print('Temperature range: {:.1f} – {:.1f} K'.format(temp_range.min(), temp_range.max()))
print('Size range: {:.3e} – {:.3e} µm'.format(sizes.min(), sizes.max()))
print('Distance range: {:.4f} – {:.1f} AU'.format(therm_dist.min(), therm_dist.max()))


## 4. Interpolate temperatures at chosen distances

The r(a,Td) function is then numerically reversed to get Td(a,r). This is done using the `stargrain.get_temperature(distances)` to get equilibrium temperatures for all sizes at specified distances.

Source: Lebreton et al. 2013

In [ ]:
distances = np.linspace(0.01, 10, 2000) #A range of r for the calculation of Td

temperatures = temperature.get_temperature(distances)

print('Temperatures shape:', temperatures.shape)
print('Temperature range: {:.2f} – {:.2f} K'.format(temperatures.min(), temperatures.max()))



## 3. Visualize raw thermal distance grid
Rows: sizes; Columns: temperatures.

In [ ]:
import matplotlib.colors as mcolors

plt.figure(figsize=(6,4))
plt.imshow(therm_dist, aspect='auto', origin='lower',
           extent=[temp_range.min(), temp_range.max(), sizes.min(), sizes.max()],
           norm=mcolors.LogNorm())
plt.colorbar(label='Distance [au]')
plt.xlabel('Temperature [K]')
plt.ylabel('Grain size [µm]')
plt.title('Thermal distance grid')
# plt.xscale('log')
plt.yscale('log')
plt.tight_layout()

## 5. Plot temperature vs distance for selected sizes

In [ ]:
idx_sample = np.linspace(0, len(sizes)-1, 5, dtype=int)
plt.figure(figsize=(7,4))
for i in idx_sample:
    plt.loglog(distances, temperatures[i], label=f'a={sizes[i]:.2f} µm')
plt.xlabel('Distance [au]')
plt.ylabel('Temperature [K]')
plt.title('Equilibrium temperature profiles')
plt.legend()
plt.tight_layout()

## 6. Contour plot (size vs distance)
Log-size vs distance colored by temperature.

Be careful: To plot this you must have previously calculated the `stargrain.get_temperature` method

In [ ]:
temperature.plot_temperatures(min_size=None, max_size=None, min_dist=None, max_dist=0.5)

## 7. Notes
- Grid resolution controlled by `N_temp` in `GrainStar`.
- Sublimation temperature (`Tsub`) limits high-T region.
- Thermal distance grid cached on disk (NPZ) for reuse.
- Modify grain optical properties (sizes / wavelengths) via `general.yaml` then recompute.